In [195]:
!jupyter nbconvert --to notebook --execute preprocess.ipynb --inplace

[NbConvertApp] Converting notebook preprocess.ipynb to notebook
[NbConvertApp] Writing 130879 bytes to preprocess.ipynb


In [196]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from pulp import *
from scipy.stats import poisson, norm
from pulp import LpProblem, LpMaximize, LpVariable, LpStatus, lpSum, value, PULP_CBC_CMD

In [197]:
OUT_CSV = "fantasy_enriched.csv"



| Position        | Point Source         | Points | Description                                                                 |
| --------------- | -------------------- | -----: | --------------------------------------------------------------------------- |
| **All Players** | Appearance (<60 min) |     +1 | Plays any minutes up to 59:59                                               |
| **All Players** | Appearance (60+ min) |     +1 | Additional point for reaching 60+ minutes (total appearance = **2 points**) |
| **All Players** | Assist               |     +3 | Provides an assist                                                          |
| **All Players** | Yellow Card          |     -1 | Receives a yellow card                                                      |
| **All Players** | Red Card             |     -2 | Receives a red card                                                         |
| **All Players** | Own Goal             |     -2 | Scores an own goal                                                          |
| **All Players** | Winning a Penalty    |     +2 | Wins a penalty for their team                                               |
| **All Players** | Conceding a Penalty  |     -1 | Commits a foul leading to a penalty                                         |

| Position       | Point Source                  | Points | Description                                        |
| -------------- | ----------------------------- | -----: | -------------------------------------------------- |
| **Goalkeeper** | Clean Sheet                   |     +5 | Plays 60+ minutes and team keeps a clean sheet     |
| **Goalkeeper** | First Goal Conceded           |      0 | No deduction for conceding the first goal          |
| **Goalkeeper** | Each Additional Goal Conceded |     -1 | Every goal conceded after the first                |
| **Goalkeeper** | Goal Scored                   |     +9 | Scores a goal                                      |
| **Goalkeeper** | Penalty Save                  |     +3 | Saves a penalty during normal play (not shootouts) |
| **Goalkeeper** | Every 3 Saves                 |     +1 | Earns 1 point for every 3 saves                    |

| Position     | Point Source                  | Points | Description                                    |
| ------------ | ----------------------------- | -----: | ---------------------------------------------- |
| **Defender** | Clean Sheet                   |     +5 | Plays 60+ minutes and team keeps a clean sheet |
| **Defender** | First Goal Conceded           |      0 | No deduction for conceding the first goal      |
| **Defender** | Each Additional Goal Conceded |     -1 | Every goal conceded after the first            |
| **Defender** | Goal Scored                   |     +7 | Scores a goal                                  |

| Position       | Point Source                | Points | Description                                    |
| -------------- | --------------------------- | -----: | ---------------------------------------------- |
| **Midfielder** | Clean Sheet                 |     +1 | Plays 60+ minutes and team keeps a clean sheet |
| **Midfielder** | Goal Scored                 |     +6 | Scores a goal                                  |
| **Midfielder** | Every 3 Tackles             |     +1 | Earns 1 point for every 3 tackles              |
| **Midfielder** | Every 2 Big Chances Created |     +1 | Earns 1 point for every 2 big chances created  |

| Position    | Point Source            | Points | Description                               |
| ----------- | ----------------------- | -----: | ----------------------------------------- |
| **Forward** | Goal Scored             |     +5 | Scores a goal                             |
| **Forward** | Every 2 Shots on Target |     +1 | Earns 1 point for every 2 shots on target |

| Position                | Point Source          | Points | Description                                                                                                                                                                        |
| ----------------------- | --------------------- | -----: | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Bonus (All Players)** | Direct Free-Kick Goal |     +1 | Additional bonus for scoring directly from a free kick (on top of goal points)                                                                                                     |
| **Bonus (All Players)** | Scouting Bonus        |     +2 | Scores **more than 4 base fantasy points** in a match **and** is selected by **<5%** of fantasy teams. Captaincy and booster points do **not** count toward the 4-point threshold. |



## Core identity / player info

| Column | Type | Description |
|---|---|---|
| `fifa_id` | int | Unique FIFA fantasy player ID |
| `name` | str | Player name |
| `squad_id` | int | Internal squad/nation ID |
| `team` | str | National team |
| `position` | str | DEF / MID / FWD / GK |
| `price` | float | Fantasy price |
| `status` | str | Availability status |
| `total_points` | int | Total tournament points |
| `avg_points` | float | Average points per match |
| `matches_played` | int | Matches with participation |
| `percent_selected` | float | Selection rate (%) |
| `next_fixture` | int | Next match ID |


## Round fantasy points

| Column | Type | Description |
|---|---|---|
| `round_1_points` | int | Points in round 1 |
| `round_2_points` | int | Points in round 2 |
| `round_3_points` | int | Points in round 3 |
| `round_4_points` | int | Points in round 4 |


## Betting / scorer mapping

| Column | Type | Description |
|---|---|---|
| `betano_matched_name` | str | Matched Betano market name |
| `betano_match_score` | float | Name matching confidence (0–100) |
| `anytime_scorer_prob` | float | Probability player scores anytime |
| `first_scorer_prob` | float | Probability scores first goal |
| `last_scorer_prob` | float | Probability scores last goal |


## Match context

| Column | Type | Description |
|---|---|---|
| `match_home` | str | Home team |
| `match_away` | str | Away team |
| `is_home` | bool | Player team is home |
| `opponent` | str | Opponent team |


## Match probabilities 

| Column | Type | Description |
|---|---|---|
| `match_home_win_prob` | float | Home win probability |
| `match_draw_prob` | float | Draw probability |
| `match_away_win_prob` | float | Away win probability |
| `match_btts_prob` | float | Both teams score probability |
| `match_over_25_prob` | float | Over 2.5 goals probability |
| `match_over_35_prob` | float | Over 3.5 goals probability |
| `team_score_prob` | float | Team scores ≥1 goal probability |
| `team_score_2_prob` | float | Team scores ≥2 goals probability |
| `team_cs_prob` | float | Team clean sheet probability |
| `opp_score_prob` | float | Opponent scores ≥1 goal probability |
| `opp_cs_prob` | float | Opponent clean sheet probability |
| `team_over_05_prob` | float | Team scores ≥1 goal probability |
| `team_over_15_prob` | float | Team scores ≥2 goals probability |
| `team_qualify_prob` | float | Team advances probability |
| `team_win2_prob` | float | Team wins by 2+ goals probability |
| `opp_over_05_prob` | float | Opponent scores ≥1 goal probability |


## Raw cumulative stats

| Column | Type | Description |
|---|---|---|
| `stat_GS` | int | Goals |
| `stat_AS` | int | Assists |
| `stat_CS` | int | Clean sheets |
| `stat_GC` | int | Goals conceded |
| `stat_MP` | int | Minutes played |
| `stat_YC` | int | Yellow cards |
| `stat_RC` | int | Red cards |
| `stat_ST` | int | Shots on target |
| `stat_SB` | int | Shots blocked |
| `stat_CC` | int | Chances created |
| `stat_PS` | int | Penalties saved |
| `stat_T` | int | Tackles |
| `stat_S` | int | Saves |
| `stat_SXI` | int | Starts (XI appearances) |
| `stat_OG` | int | Own goals |
| `stat_PC` | int | Penalties committed |
| `stat_PW` | int | Penalties won |
| `stat_FK` | int | Free kicks won |



## Round-by-round stats (1–4)

| Pattern | Type | Description |
|---|---|---|
| `round_i_SXI` | int | Started (0/1) |
| `round_i_MP` | int | Minutes |
| `round_i_AS` | int | Assists |
| `round_i_YC` | int | Yellow cards |
| `round_i_RC` | int | Red cards |
| `round_i_OG` | int | Own goals |
| `round_i_PW` | int | Penalties won |
| `round_i_PC` | int | Penalties committed |
| `round_i_CS` | int | Clean sheets |
| `round_i_GS` | int | Goals |
| `round_i_GC` | int | Goals conceded |
| `round_i_PS` | int | Penalties saved |
| `round_i_T` | int | Tackles |
| `round_i_CC` | int | Chances created |
| `round_i_ST` | int | Shots on target |
| `round_i_FK` | int | Free kicks |
| `round_i_S` | int | Saves |
| `round_i_SB` | int | Shots blocked |



## External matching / enrichment

| Column | Type | Description |
|---|---|---|
| `clubstats_matched_player` | str | External player match |
| `clubs_match_dist` | float | Matching distance score |



## Form / usage trends

| Column | Type | Description |
|---|---|---|
| `started_last_match` | int | Started last match |
| `starts_last_3` | int | Starts last 3 matches |
| `starts_last_5` | int | Starts last 5 matches |
| `start_rate` | float | Start frequency |
| `minutes_last_3_avg` | float | Avg minutes last 3 |
| `minutes_last_5_avg` | float | Avg minutes last 5 |



## Performance rates

| Column | Type | Description |
|---|---|---|
| `goal_rate` | float | Goals per minute |
| `goals_per_start` | float | Goals per start |
| `shots_on_target_per90` | float | SOT per 90 |
| `goals_per_shot_on_target` | float | Conversion rate |
| `assist_rate` | float | Assists per minute |
| `chance_created_per90` | float | Chances per 90 |
| `tackles_per90` | float | Tackles per 90 |
| `cc_per90` | float | Chances created per 90 |
| `sot_per90` | float | Shots on target per 90 |
| `saves_per90` | float | Saves per 90 |
| `yc_per90` | float | Yellow cards per 90 |
| `rc_per90` | float | Red cards per 90 |
| `penalty_conceded_rate` | float | Penalties conceded per 90 |
| `own_goal_rate` | float | Own goals per 90 |
| `penalties_won_per90` | float | Penalties won per 90 |



## Recent / streak features

| Column | Type | Description |
|---|---|---|
| `recent_goals_last3` | int | Goals last 3 matches |
| `goal_streak` | int | Consecutive scoring matches |
| `recent_assists_last3` | int | Assists last 3 matches |
| `recent_chances_created_per90` | float | Recent creation rate |
| `recent_cs_rate` | float | Clean sheet rate |
| `recent_tackles_per90` | float | Recent tackles |
| `recent_cc_per90` | float | Recent chances created |
| `recent_sot_per90` | float | Recent shots on target |
| `recent_saves_per90` | float | Recent saves |
| `recent_penalties_won` | int | Penalties won last 3 |



## Expected value features

| Column | Type | Description |
|---|---|---|
| `goal_expectation` | float | Scoring proxy |
| `goal_expectation2` | float | Scoring proxy (2+ goals) |
| `assist_expectation` | float | Assist proxy |
| `cs_expectation` | float | Clean sheet proxy |
| `expected_minutes` | float | Expected minutes |
| `expected_gc_penalty` | float | Defensive penalty |
| `expected_tackle_points` | float | Tackles contribution |
| `expected_cc_points` | float | Chance creation points |
| `expected_sot_points` | float | Shot contribution points |
| `expected_save_points` | float | Save contribution points |



## Fantasy points dynamics

| Column | Type | Description |
|---|---|---|
| `points_last1` | int | Last match points |
| `points_last3_avg` | float | Avg last 3 |
| `points_last5_avg` | float | Avg last 5 |
| `weighted_points` | float | Recency weighted |
| `rolling_std_points` | float | Variance last 5 |
| `rolling_max_points` | int | Max last 5 |



## Team-position ranking features

| Column | Type | Description |
|---|---|---|
| `price_rank_team_position` | float | Price rank |
| `selected_rank_team_position` | float | Selection rank |
| `points_rank_team_position` | float | Points rank |
| `minutes_rank_team_position` | float | Minutes rank |
| `starts_rank_team_position` | float | Starts rank |
| `anytime_rank_team_position` | float | Scoring odds rank |
| `shots_rank_team_position` | float | Shooting rank |
| `cc_rank_team_position` | float | Chance creation rank |
| `tackles_rank_team_position` | float | Tackles rank |

In [198]:
df= pd.read_csv("fantasy_enriched.csv")

In [199]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [200]:
df = df[df["status"] == "playing"].copy()

In [201]:
df.loc[df["name"] == "Johan Manzambi", "status"] = "injured"  # :(
df.loc[df["name"] == "Nicolás Tagliafico", "status"] = "injured"  # :(


In [202]:
# probability of playing a minute

raw_any = (
    0.2* df["minutes_rank_team_position"]
  + 0.10 * df["stat_MP"]        # minutes needs to be very team stratified since while for other point sources players are all eligible
  + 0.15 * df["starts_rank_team_position"]
  + 0.10 * df["price_rank_team_position"]     # for minutes its a team stratified signal.
  + 0.1 * df["selected_rank_team_position"]
  + 0.25 * df["minutes_last_3_avg"]
  + 0.1 * df["has_betano_match"] # having odd at all means likeliness of playing is higher.

)

df["prob_plays_any"] = sigmoid((raw_any - 0.5) * 6) 

In [203]:
df.loc[
    :,
    [
        "prob_plays_any",
        "name",
        "team",
        "position",
        "stat_MP",
        "minutes_rank_team_position",
        "starts_rank_team_position",
        "selected_rank_team_position",
        "price_rank_team_position",
        "anytime_rank_team_position",
    ],
].sort_values("prob_plays_any", ascending=False).head(30)

,prob_plays_any,name,team,position,stat_MP,minutes_rank_team_position,starts_rank_team_position,selected_rank_team_position,price_rank_team_position,anytime_rank_team_position
10,0.937027,Emiliano Martínez,Argentina,GK,1.000000,1.000000,0.666667,1.000000,1.000000,1.000000
27,0.932474,Harry Kane,England,FWD,0.946377,1.000000,0.625000,1.000000,1.000000,0.750000
5,0.930966,Lionel Messi,Argentina,FWD,0.898551,1.000000,0.625000,1.000000,1.000000,1.000000
19,0.922964,Alexis Mac Allister,Argentina,MID,0.911594,1.000000,0.545455,1.000000,0.909091,0.090909
48,0.917866,Kylian Mbappé,France,FWD,0.881159,1.000000,0.666667,1.000000,1.000000,1.000000
22,0.913589,Marc Guéhi,England,DEF,0.830435,1.000000,0.555556,0.877778,0.888889,0.333333
40,0.908552,Jude Bellingham,England,MID,0.876812,0.900000,0.550000,1.000000,0.900000,0.900000
43,0.907682,Dayot Upamecano,France,DEF,0.892754,1.000000,0.555556,1.000000,0.833333,0.611111
82,0.906560,Lisandro Martínez,Argentina,DEF,0.843478,1.000000,0.562500,1.000000,0.750000,0.250000
72,0.904190,Mikel Oyarzabal,Spain,FWD,0.786957,1.000000,0.625000,1.000000,1.000000,1.000000


In [204]:
# probability of starting / playing 60 minutes

raw_any = (
    0.25 * df["minutes_rank_team_position"]
  + 0.05 * df["starts_rank_team_position"]
  + 0.20 * df["stat_MP"]
  + 0.05 * df["selected_rank_team_position"]
  + 0.40 * df["minutes_last_3_avg"]
  + 0.05 * df["has_betano_match"] # having odd at all means likeliness of playing is higher.

)

df["prob_plays_60min"] = sigmoid((raw_any - 0.5) * 6)


In [205]:
cols = [
    "prob_plays_60min",
    "name",
    "team",
    "position",
    "minutes_rank_team_position",
    "starts_rank_team_position",
    "price_rank_team_position",
    "selected_rank_team_position",
    "minutes_last_3_avg",
    "round_5_MP",
    "anytime_rank_team_position",
]

df[cols].sort_values("prob_plays_60min", ascending=False).head(30)

,prob_plays_60min,name,team,position,minutes_rank_team_position,starts_rank_team_position,price_rank_team_position,selected_rank_team_position,minutes_last_3_avg,round_5_MP,anytime_rank_team_position
10,0.947846,Emiliano Martínez,Argentina,GK,1.000000,0.666667,1.000000,1.000000,1.000000,1.000000,1.000000
27,0.943489,Harry Kane,England,FWD,1.000000,0.625000,1.000000,1.000000,0.996667,0.988889,0.750000
5,0.940797,Lionel Messi,Argentina,FWD,1.000000,0.625000,1.000000,1.000000,1.000000,1.000000,1.000000
19,0.940338,Alexis Mac Allister,Argentina,MID,1.000000,0.545455,0.909091,1.000000,1.000000,1.000000,0.090909
22,0.932549,Marc Guéhi,England,DEF,1.000000,0.555556,0.888889,0.877778,1.000000,1.000000,0.333333
33,0.927427,Jordan Pickford,England,GK,1.000000,0.666667,1.000000,1.000000,1.000000,1.000000,0.666667
82,0.926690,Lisandro Martínez,Argentina,DEF,1.000000,0.562500,0.750000,1.000000,0.940000,1.000000,0.250000
43,0.923999,Dayot Upamecano,France,DEF,1.000000,0.555556,0.833333,1.000000,0.900000,1.000000,0.611111
38,0.923895,Elliot Anderson,England,MID,1.000000,0.550000,0.300000,0.560000,0.950000,0.833333,0.300000
40,0.923803,Jude Bellingham,England,MID,0.900000,0.550000,0.900000,1.000000,0.970000,1.000000,0.900000


There is a phenomenon where a player will be overvalued, such as Lukaku here for belgium. He has not played much, but since Belgium only has 3 forward and only him has significative minutes he ranks number 1 for all ranks in BEL,FWD. Need to include more general sums and beware with manual selection.

In [206]:
df["lambda_goal"] = -np.log(
    (1 - df["anytime_scorer_prob"]).clip(lower=0.001)
)

lambda_adj = (
    0.70 * df["lambda_goal"]
  + 0.1 * df["recent_goals_last3"]
  + 0.02 * df["team_score_2_prob"]
  + 0.1 * df["stat_GS"]
  + 0.05 * df["stat_ST"]
  + 0.03 * df["price"]
)

df["prob_scores"] = (
    lambda_adj.clip(lower=0)
    * df["prob_plays_60min"]
    * 2
)

df.loc[df["position"] == "GK", "prob_scores"] = 0.
df.loc[df["position"] == "DEF", "prob_scores"] = df["prob_scores"] * 0.7

In [207]:
cols = [
    "name",
    "team",
    "position",
    "prob_scores",
    "goal_expectation",
    "anytime_scorer_prob",
    "recent_goals_last3",
    "recent_sot_per90",
    "goal_rate",
    "shots_on_target_per90",
    "stat_GS",
    "stat_ST",
    "price_rank_team_position",
]

df[cols].sort_values("prob_scores", ascending=False).head(30)

,name,team,position,prob_scores,goal_expectation,anytime_scorer_prob,recent_goals_last3,recent_sot_per90,goal_rate,shots_on_target_per90,stat_GS,stat_ST,price_rank_team_position
48,Kylian Mbappé,France,FWD,1.565696,0.474829,0.5814,0.50,0.245136,0.789474,0.656250,1.000,1.000000,1.000000
27,Harry Kane,England,FWD,1.026174,0.293124,0.4132,0.25,0.105351,0.551302,0.385911,0.750,0.631579,1.000000
5,Lionel Messi,Argentina,FWD,1.023263,0.246426,0.3817,0.25,0.105000,0.774194,0.609677,1.000,0.947368,1.000000
72,Mikel Oyarzabal,Spain,FWD,0.826490,0.256120,0.3636,0.25,0.130165,0.552486,0.425414,0.625,0.578947,1.000000
49,Ousmane Dembélé,France,MID,0.822799,0.289602,0.3546,0.25,0.119318,0.543478,0.304348,0.625,0.421053,1.000000
40,Jude Bellingham,England,MID,0.776769,0.166922,0.2353,1.00,0.180412,0.595041,0.381818,0.750,0.578947,0.900000
61,Michael Olise,France,MID,0.590169,0.263467,0.3226,0.00,0.000000,0.000000,0.187500,0.000,0.263158,0.909091
70,Lamine Yamal,Spain,MID,0.534093,0.190399,0.2703,0.00,0.155556,0.121212,0.424242,0.125,0.526316,1.000000
6,Julián Alvarez,Argentina,FWD,0.521338,0.169922,0.2632,0.25,0.105351,0.120240,0.210421,0.125,0.263158,0.500000
30,Anthony Gordon,England,MID,0.423561,0.175151,0.2469,0.25,0.090129,0.150376,0.210526,0.125,0.210526,0.450000


In [208]:
prob_assists = (
    0.5 * df["chance_created_per90"]
  + 0.15 * df["team_score_2_prob"]
  + 0.1 * df["prob_scores"]
  + 0.1 * df["assist_rate"]
  + 0.15 * df["stat_AS"]
)

df["lambda_assist"] = -np.log(
    (1 - prob_assists).clip(lower=0.001)
)

df["expected_assists"] = df["lambda_assist"].clip(lower=0) * df["prob_plays_60min"]  * 1.5

In [209]:
df.loc[df["position"] == "GK", "expected_assists"] = df["expected_assists"] * 0.1
df.loc[df["position"] == "DEF", "expected_assists"] = df["expected_assists"] * 0.9

In [210]:
cols = [
    "name",
    "team",
    "position",
    "expected_assists",
    "assist_expectation",
    "assist_rate",
    "chance_created_per90",
    "team_score_2_prob",
    "recent_assists_last3",
    "prob_plays_60min",
    "stat_AS",
]

df[cols].sort_values("expected_assists", ascending=False).head(30)

,name,team,position,expected_assists,assist_expectation,assist_rate,chance_created_per90,team_score_2_prob,recent_assists_last3,prob_plays_60min,stat_AS
5,Lionel Messi,Argentina,FWD,2.189598,0.749729,0.116129,1.000000,0.3610,1.00,0.940797,0.8
61,Michael Olise,France,MID,1.509368,0.656277,0.160714,0.691964,0.6667,0.00,0.904956,1.0
49,Ousmane Dembélé,France,MID,0.828016,0.399473,0.065217,0.421196,0.6667,0.00,0.897456,0.4
80,Dani Olmo,Spain,MID,0.796605,0.452829,0.128571,0.553571,0.4405,0.50,0.824477,0.6
48,Kylian Mbappé,France,FWD,0.748039,0.120893,0.088816,0.127467,0.6667,0.25,0.917843,0.6
40,Jude Bellingham,England,MID,0.642393,0.316592,0.029752,0.384298,0.4545,0.00,0.923803,0.2
30,Anthony Gordon,England,MID,0.623785,0.320030,0.135338,0.388471,0.4545,0.25,0.792476,0.6
39,Declan Rice,England,MID,0.561607,0.409269,0.038462,0.496795,0.4545,0.00,0.809831,0.2
68,Marc Cucurella,Spain,DEF,0.559487,0.301886,0.085714,0.369048,0.4405,0.00,0.919135,0.6
28,Bukayo Saka,England,MID,0.523140,0.478247,0.202247,0.580524,0.4545,0.25,0.521574,0.6


In [211]:
position_yc_modifier = {
    "GK": 0.15,
    "FWD": 0.35,
    "DEF": 0.45,
    "MID": 0.55,
}

df["position_yc_modifier"] = df["position"].map(position_yc_modifier)

raw_yc = (
    0.4 * df["yc_per90"]
  + 0.25 * df["tackles_per90"]
  + 0.20 * df["opp_over_05_prob"]
  + 0.05 * df["rc_per90"]
  + 0.1 * df["position_yc_modifier"]
)

df["prob_yellow_card"] = raw_yc * df["prob_plays_60min"]

In [212]:
cols = [
    "name",
    "team",
    "position",
    "prob_yellow_card",
    "yc_per90",
    "stat_YC",
    "tackles_per90",
    "recent_tackles_per90",
    "opp_over_05_prob",
    "prob_plays_60min",
    "rc_per90",
]

df[cols].sort_values("prob_yellow_card", ascending=False).head(30)

,name,team,position,prob_yellow_card,yc_per90,stat_YC,tackles_per90,recent_tackles_per90,opp_over_05_prob,prob_plays_60min,rc_per90
38,Elliot Anderson,England,MID,0.288535,0.028892,0.5,0.182986,0.105263,1.000000,0.923895,0.0
40,Jude Bellingham,England,MID,0.278630,0.029752,0.5,0.138843,0.051546,1.000000,0.923803,0.0
39,Declan Rice,England,MID,0.241807,0.076923,1.0,0.051282,0.027650,1.000000,0.809831,0.0
22,Marc Guéhi,England,DEF,0.240192,0.031414,0.5,0.000000,0.000000,1.000000,0.932549,0.0
27,Harry Kane,England,FWD,0.228222,0.000000,0.0,0.027565,0.020067,1.000000,0.943489,0.0
30,Anthony Gordon,England,MID,0.216977,0.000000,0.0,0.075188,0.051502,1.000000,0.792476,0.0
24,Ezri Konsa,England,DEF,0.209544,0.000000,0.0,0.032316,0.015228,1.000000,0.827978,0.0
21,Nico O'Reilly,England,DEF,0.209110,0.038961,0.5,0.077922,0.035714,1.000000,0.746648,0.0
33,Jordan Pickford,England,GK,0.199397,0.000000,0.0,0.000000,0.000000,1.000000,0.927427,0.0
25,John Stones,England,DEF,0.190312,0.000000,0.0,0.036364,0.012397,1.000000,0.748993,0.0


In [213]:
raw_pw = (
    0.10 * df["stat_PW"]                    
  + 0.3 * df["anytime_scorer_prob"]      
  + 0.10 * df["chance_created_per90"]     
  + 0.2 * df["stat_CC"]                  # recent form in 
  + 0.2 * df["team_score_2_prob"]        # team expected to attack heavily
  + 0.10 * df["price"] # quality proxy
)

df["prob_pen_won"] = 0.4 * sigmoid((raw_pw - 0.65) * 5) * df["prob_plays_60min"]

In [214]:
cols = [
    "name",
    "team",
    "position",
    "prob_pen_won",
    "stat_PW",
    "anytime_scorer_prob",
    "assist_expectation",
    "chance_created_per90",
    "team_score_2_prob",
    "price_rank_team_position",
    "prob_plays_60min",
]

df[cols].sort_values("prob_pen_won", ascending=False).head(30)

,name,team,position,prob_pen_won,stat_PW,anytime_scorer_prob,assist_expectation,chance_created_per90,team_score_2_prob,price_rank_team_position,prob_plays_60min
5,Lionel Messi,Argentina,FWD,0.155116,0.0,0.3817,0.749729,1.000000,0.3610,1.000000,0.940797
48,Kylian Mbappé,France,FWD,0.136676,1.0,0.5814,0.120893,0.127467,0.6667,1.000000,0.917843
61,Michael Olise,France,MID,0.121766,0.0,0.3226,0.656277,0.691964,0.6667,0.909091,0.904956
49,Ousmane Dembélé,France,MID,0.098978,0.0,0.3546,0.399473,0.421196,0.6667,1.000000,0.897456
70,Lamine Yamal,Spain,MID,0.082865,1.0,0.2703,0.128073,0.156566,0.4405,1.000000,0.895495
27,Harry Kane,England,FWD,0.069334,0.0,0.4132,0.097773,0.118683,0.4545,1.000000,0.943489
30,Anthony Gordon,England,MID,0.068582,1.0,0.2469,0.320030,0.388471,0.4545,0.450000,0.792476
40,Jude Bellingham,England,MID,0.064758,0.0,0.2353,0.316592,0.384298,0.4545,0.900000,0.923803
89,Jules Koundé,France,DEF,0.061565,0.0,0.1053,0.244196,0.257475,0.6667,1.000000,0.901110
80,Dani Olmo,Spain,MID,0.055217,0.0,0.1923,0.452829,0.553571,0.4405,0.727273,0.824477


In [215]:
raw_cs = (
    0.90 * df["team_cs_prob"]            # strongest signal
  + 0.05 * df["stat_CS"]               # tournament history
  + 0.05 * (1 - df["stat_GC"])         # fewer goals conceded is better
)

base_cs = sigmoid((raw_cs - 0.5) * 6)

df["prob_clean_sheet"] = base_cs * df["prob_plays_60min"]

df.loc[df["position"] == "FWD", "prob_clean_sheet"] = 0.0

In [216]:
cols = [
    "name",
    "team",
    "position",
    "prob_clean_sheet",
    "team_cs_prob",
    "prob_plays_60min",
    "stat_CS",
    "stat_GC",
    "minutes_rank_team_position",
    "tackles_per90",
]

df[cols].sort_values("prob_clean_sheet", ascending=False).head(30)

,name,team,position,prob_clean_sheet,team_cs_prob,prob_plays_60min,stat_CS,stat_GC,minutes_rank_team_position,tackles_per90
68,Marc Cucurella,Spain,DEF,0.332747,0.3544,0.919135,0.857143,0.125,0.937500,0.066667
79,Rodri,Spain,MID,0.330905,0.3544,0.914046,0.857143,0.125,1.000000,0.220096
69,Pau Cubarsí,Spain,DEF,0.330449,0.3544,0.912787,0.857143,0.125,0.937500,0.047619
76,Unai Simón,Spain,GK,0.327680,0.3544,0.905137,0.857143,0.125,1.000000,0.000000
70,Lamine Yamal,Spain,MID,0.324189,0.3544,0.895495,0.857143,0.125,0.909091,0.096970
66,Aymeric Laporte,Spain,DEF,0.322752,0.3544,0.891525,0.857143,0.125,0.750000,0.047695
80,Dani Olmo,Spain,MID,0.298479,0.3544,0.824477,0.857143,0.125,0.727273,0.100000
65,Pedro Porro,Spain,DEF,0.278274,0.3544,0.812551,0.571429,0.125,0.625000,0.162162
63,Álex Baena,Spain,MID,0.263349,0.3544,0.747760,0.714286,0.125,0.636364,0.116505
81,Pedri,Spain,MID,0.256439,0.3544,0.673458,1.000000,0.000,0.818182,0.136054


In [217]:
position_rc_modifier = {
    "GK": 0.04,
    "FWD": 0.08,
    "DEF": 0.12,
    "MID": 0.14,
}

df["position_rc_modifier"] = df["position"].map(position_rc_modifier)

raw_rc = (
    0.10 * df["position_rc_modifier"]   # strongest prior
  + 0.25 * df["rc_per90"]              # direct history
  + 0.15 * df["tackles_per90"]         # physical involvement
  + 0.25 * df["prob_yellow_card"]      # disciplinary tendency
  + 0.25 * df["opp_over_05_prob"]      # more defending -> more challenges
)

df["prob_red_card"] = 0.1 * sigmoid((raw_rc - 0.5) * 5) * df["prob_plays_60min"]

In [218]:
cols = [
    "name",
    "team",
    "position",
    "prob_red_card",
    "position_rc_modifier",
    "rc_per90",
    "tackles_per90",
    "prob_yellow_card",
    "opp_over_05_prob",
    "prob_plays_any",
]

df[cols].sort_values("prob_red_card", ascending=False).head(30)

,name,team,position,prob_red_card,position_rc_modifier,rc_per90,tackles_per90,prob_yellow_card,opp_over_05_prob,prob_plays_any
38,Elliot Anderson,England,MID,0.031024,0.14,0.0,0.182986,0.288535,1.000000,0.855417
40,Jude Bellingham,England,MID,0.030091,0.14,0.0,0.138843,0.278630,1.000000,0.908552
27,Harry Kane,England,FWD,0.027193,0.08,0.0,0.027565,0.228222,1.000000,0.932474
22,Marc Guéhi,England,DEF,0.027152,0.12,0.0,0.000000,0.240192,1.000000,0.913589
33,Jordan Pickford,England,GK,0.025295,0.04,0.0,0.000000,0.199397,1.000000,0.888342
39,Declan Rice,England,MID,0.024432,0.14,0.0,0.051282,0.241807,1.000000,0.783292
24,Ezri Konsa,England,DEF,0.023868,0.12,0.0,0.032316,0.209544,1.000000,0.825781
30,Anthony Gordon,England,MID,0.023690,0.14,0.0,0.075188,0.216977,1.000000,0.777465
21,Nico O'Reilly,England,DEF,0.022042,0.12,0.0,0.077922,0.209110,1.000000,0.781610
25,John Stones,England,DEF,0.021269,0.12,0.0,0.036364,0.190312,1.000000,0.723847


In [219]:
position_og_modifier = {
    "GK": 0.05,
    "FWD": 0.02,
    "MID": 0.04,
    "DEF": 0.08,
}

df["position_og_modifier"] = df["position"].map(position_og_modifier)

raw_og = (
    0.4 * df["position_og_modifier"]
  + 0.5 * df["opp_over_05_prob"]
  + 0.10 * (1 - df["stat_GC"])
)

df["prob_own_goal"] = 0.03 * sigmoid((raw_og - 0.5) * 5) * df["prob_plays_60min"]

In [220]:
cols = [
    "name",
    "team",
    "position",
    "prob_own_goal",
    "position_og_modifier",
    "own_goal_rate",
    "opp_over_05_prob",
    "tackles_per90",
    "stat_GC",
    "prob_plays_any",
]

df[cols].sort_values("prob_own_goal", ascending=False).head(30)

,name,team,position,prob_own_goal,position_og_modifier,own_goal_rate,opp_over_05_prob,tackles_per90,stat_GC,prob_plays_any
22,Marc Guéhi,England,DEF,0.015968,0.08,0.0,1.000000,0.000000,0.750,0.913589
33,Jordan Pickford,England,GK,0.014606,0.05,0.0,1.000000,0.000000,1.000,0.888342
27,Harry Kane,England,FWD,0.014435,0.02,0.0,1.000000,0.027565,1.000,0.932474
38,Elliot Anderson,England,MID,0.014412,0.04,0.0,1.000000,0.182986,1.000,0.855417
40,Jude Bellingham,England,MID,0.014411,0.04,0.0,1.000000,0.138843,1.000,0.908552
30,Anthony Gordon,England,MID,0.013468,0.04,0.0,1.000000,0.075188,0.625,0.777465
24,Ezri Konsa,England,DEF,0.013411,0.08,0.0,1.000000,0.032316,1.000,0.825781
39,Declan Rice,England,MID,0.013388,0.04,0.0,1.000000,0.051282,0.750,0.783292
25,John Stones,England,DEF,0.012825,0.08,0.0,1.000000,0.036364,0.750,0.723847
21,Nico O'Reilly,England,DEF,0.012094,0.08,0.0,1.000000,0.077922,1.000,0.781610


In [221]:
position_pc_modifier = {
    "GK": 0.08,
    "FWD": 0.01,
    "MID": 0.05,
    "DEF": 0.14,
}

df["position_pc_modifier"] = df["position"].map(position_pc_modifier)

raw_pc = (
    0.20 * df["position_pc_modifier"]
  + 0.15 * df["penalty_conceded_rate"]
  + 0.15 * df["tackles_per90"]
  + 0.35 * df["prob_yellow_card"]
  + 0.15 * df["opp_over_05_prob"]
)

df["prob_penalty_committed"] = 0.15 * sigmoid((raw_pc - 0.5) * 5) * df["prob_plays_60min"]

In [222]:
cols = [
    "name",
    "team",
    "position",
    "prob_penalty_committed",
    "position_pc_modifier",
    "penalty_conceded_rate",
    "tackles_per90",
    "prob_yellow_card",
    "opp_over_05_prob",
    "prob_plays_any",
]

df[cols].sort_values("prob_penalty_committed", ascending=False).head(30)

,name,team,position,prob_penalty_committed,position_pc_modifier,penalty_conceded_rate,tackles_per90,prob_yellow_card,opp_over_05_prob,prob_plays_any
27,Harry Kane,England,FWD,0.035911,0.01,0.321593,0.027565,0.228222,1.000000,0.932474
38,Elliot Anderson,England,MID,0.035717,0.05,0.000000,0.182986,0.288535,1.000000,0.855417
40,Jude Bellingham,England,MID,0.034392,0.05,0.000000,0.138843,0.278630,1.000000,0.908552
22,Marc Guéhi,England,DEF,0.032637,0.14,0.000000,0.000000,0.240192,1.000000,0.913589
33,Jordan Pickford,England,GK,0.029303,0.08,0.000000,0.000000,0.199397,1.000000,0.888342
24,Ezri Konsa,England,DEF,0.028329,0.14,0.000000,0.032316,0.209544,1.000000,0.825781
39,Declan Rice,England,MID,0.027298,0.05,0.000000,0.051282,0.241807,1.000000,0.783292
21,Nico O'Reilly,England,DEF,0.026212,0.14,0.000000,0.077922,0.209110,1.000000,0.781610
30,Anthony Gordon,England,MID,0.026188,0.05,0.000000,0.075188,0.216977,1.000000,0.777465
25,John Stones,England,DEF,0.025026,0.14,0.000000,0.036364,0.190312,1.000000,0.723847


In [223]:
df["lambda_opp"] = -np.log(df["team_cs_prob"].clip(lower=0.01))

df["prob_gc_0"] = np.exp(-df["lambda_opp"])
df["prob_gc_1"] = df["lambda_opp"] * np.exp(-df["lambda_opp"])
df["prob_gc_2"] = (df["lambda_opp"] ** 2 / 2) * np.exp(-df["lambda_opp"])
df["prob_gc_3"] = (df["lambda_opp"] ** 3 / 6) * np.exp(-df["lambda_opp"])
df["prob_gc_4plus"] = 1 - (
    df["prob_gc_0"]
    + df["prob_gc_1"]
    + df["prob_gc_2"]
    + df["prob_gc_3"]
)

# RAW expected goals conceded (true intensity)
df["expected_goals_conceded"] = df["lambda_opp"]

# Normalize ONLY as auxiliary feature (separate column)
scaler = MinMaxScaler()
df["expected_goals_conceded_norm"] = scaler.fit_transform(
    df[["expected_goals_conceded"]]
)

# Player exposure to goals conceded (correct)
df["expected_player_goals_conceded"] = (
    df["lambda_opp"] * df["prob_plays_60min"]
)

# GC penalty MUST use raw λ (correct Poisson expectation)
df["expected_gc_penalty"] = (
    -(df["lambda_opp"] - 1 + np.exp(-df["lambda_opp"]))
    * df["prob_plays_60min"]
)

df.loc[df["position"].isin(["MID", "FWD"]), [
    "expected_player_goals_conceded",
    "expected_gc_penalty"
]] = 0.0

In [224]:
cols = [
    "name",
    "team",
    "position",
    "prob_plays_60min",
    "team_cs_prob",
    "lambda_opp",
    "prob_gc_0",
    "prob_gc_1",
    "prob_gc_2",
    "prob_gc_3",
    "prob_gc_4plus",
    "expected_player_goals_conceded",
    "expected_gc_penalty",
]

df[df["position"].isin(["GK", "DEF"])][cols] \
    .sort_values("expected_gc_penalty") \
    .head(30)

,name,team,position,prob_plays_60min,team_cs_prob,lambda_opp,prob_gc_0,prob_gc_1,prob_gc_2,prob_gc_3,prob_gc_4plus,expected_player_goals_conceded,expected_gc_penalty
22,Marc Guéhi,England,DEF,0.932549,0.1833,1.696631,0.1833,0.310992,0.263820,0.149202,0.092686,1.582192,-0.820579
33,Jordan Pickford,England,GK,0.927427,0.1833,1.696631,0.1833,0.310992,0.263820,0.149202,0.092686,1.573502,-0.816072
24,Ezri Konsa,England,DEF,0.827978,0.1833,1.696631,0.1833,0.310992,0.263820,0.149202,0.092686,1.404773,-0.728563
25,John Stones,England,DEF,0.748993,0.1833,1.696631,0.1833,0.310992,0.263820,0.149202,0.092686,1.270764,-0.659062
21,Nico O'Reilly,England,DEF,0.746648,0.1833,1.696631,0.1833,0.310992,0.263820,0.149202,0.092686,1.266785,-0.656998
10,Emiliano Martínez,Argentina,GK,0.947846,0.2956,1.218748,0.2956,0.360262,0.219534,0.089186,0.035418,1.155186,-0.487523
43,Dayot Upamecano,France,DEF,0.923999,0.2906,1.235808,0.2906,0.359126,0.221905,0.091411,0.036959,1.141885,-0.486400
82,Lisandro Martínez,Argentina,DEF,0.926690,0.2956,1.218748,0.2956,0.360262,0.219534,0.089186,0.035418,1.129402,-0.476641
52,Mike Maignan,France,GK,0.905137,0.2906,1.235808,0.2906,0.359126,0.221905,0.091411,0.036959,1.118575,-0.476471
89,Jules Koundé,France,DEF,0.901110,0.2906,1.235808,0.2906,0.359126,0.221905,0.091411,0.036959,1.113599,-0.474351


In [225]:
raw_save = (
    0.35 * df["expected_goals_conceded_norm"]  # opportunity
  + 0.40 * df["saves_per90"]             # ability
  + 0.05 * df["recent_saves_per90"]      # recent form
  + 0.20 * df["price"]                   # overall GK quality
)

# Expected saves (λ)
df["expected_saves"] = (
    sigmoid((raw_save - 0.5) * 6) * 4
) * df["prob_plays_60min"]

df.loc[df["position"] != "GK", "expected_saves"] = 0.0

In [226]:
cols = [
    "name",
    "team",
    "expected_saves",
    "expected_goals_conceded",
    "saves_per90",
    "recent_saves_per90",
    "price",
    "expected_goals_conceded_norm",
]

df[df["position"] == "GK"][cols] \
    .sort_values("expected_saves", ascending=False) \
    .head(30)

,name,team,expected_saves,expected_goals_conceded,saves_per90,recent_saves_per90,price,expected_goals_conceded_norm
33,Jordan Pickford,England,3.504931,1.696631,1.000000,1.000000,0.866667,1.000000
10,Emiliano Martínez,Argentina,2.198208,1.218748,0.573913,0.555556,1.000000,0.275168
52,Mike Maignan,France,2.148224,1.235808,0.628571,0.123457,1.000000,0.301043
76,Unai Simón,Spain,1.899279,1.037329,0.698413,0.740741,1.000000,0.000000
34,Dean Henderson,England,0.203562,1.696631,0.000000,0.000000,0.466667,1.000000
35,James Trafford,England,0.203424,1.696631,0.000000,0.000000,0.333333,1.000000
11,Gerónimo Rulli,Argentina,0.084711,1.218748,0.000000,0.000000,0.666667,0.275168
53,Brice Samba,France,0.084463,1.235808,0.000000,0.000000,0.666667,0.301043
75,David Raya,Spain,0.076431,1.037329,0.000000,0.000000,1.000000,0.000000
12,Juan Musso,Argentina,0.073990,1.218748,0.000000,0.000000,0.533333,0.275168


In [227]:
raw_pen_save = (
    0.55 * df["price"]                    
  + 0.30 * df["expected_goals_conceded_norm"]
  + 0.15 * df["stat_PS"]
)

# Strong compression because penalty saves are exceptionally rare
df["prob_penalty_save"] = sigmoid((raw_pen_save - 0.5) * 3) / 20 * df["prob_plays_60min"]

df.loc[df["position"] != "GK", "prob_penalty_save"] = 0.0

In [228]:
cols = [
    "name",
    "team",
    "prob_penalty_save",
    "price",
    "expected_goals_conceded",
    "opp_over_05_prob",
    "prob_plays_60min",
    "stat_PS",
]

df[df["position"] == "GK"][cols] \
    .sort_values("prob_penalty_save", ascending=False) \
    .head(30)

,name,team,prob_penalty_save,price,expected_goals_conceded,opp_over_05_prob,prob_plays_60min,stat_PS
33,Jordan Pickford,England,0.032291,0.866667,1.696631,1.000000,0.927427,0.0
52,Mike Maignan,France,0.031903,1.000000,1.235808,0.375328,0.905137,1.0
10,Emiliano Martínez,Argentina,0.028346,1.000000,1.218748,0.343750,0.947846,0.0
76,Unai Simón,Spain,0.024322,1.000000,1.037329,0.000000,0.905137,0.0
75,David Raya,Spain,0.003620,1.000000,1.037329,0.000000,0.134703,0.0
34,Dean Henderson,England,0.003319,0.466667,1.696631,1.000000,0.122389,0.0
35,James Trafford,England,0.003283,0.333333,1.696631,1.000000,0.134703,0.0
11,Gerónimo Rulli,Argentina,0.002966,0.666667,1.218748,0.343750,0.128420,0.0
53,Brice Samba,France,0.002863,0.666667,1.235808,0.375328,0.122389,0.0
12,Juan Musso,Argentina,0.002620,0.533333,1.218748,0.343750,0.128420,0.0


In [229]:
raw_tackles = (
    0.50 * df["tackles_per90"]
  + 0.15 * df["recent_tackles_per90"]
  + 0.15 * df["expected_goals_conceded_norm"]
  + 0.20 * df["match_over_25_prob"]
)

df["lambda_tackles"] = -np.log(
    (1 - raw_tackles).clip(lower=0.001)
)

df["expected_tackles"] = (
    df["lambda_tackles"].clip(lower=0)
    * df["prob_plays_60min"] * 3
)

df.loc[df["position"] != "MID", "expected_tackles"] = 0.0

In [230]:
cols = [
    "name",
    "team",
    "expected_tackles",
    "tackles_per90",
    "recent_tackles_per90",
    "expected_goals_conceded",
    "match_over_25_prob",
    "price",
    "prob_plays_60min",
]

df[df["position"] == "MID"][cols] \
    .sort_values("expected_tackles", ascending=False) \
    .head(30)

,name,team,expected_tackles,tackles_per90,recent_tackles_per90,expected_goals_conceded,match_over_25_prob,price,prob_plays_60min
38,Elliot Anderson,England,1.342824,0.182986,0.105263,1.696631,0.6335,0.285714,0.923895
40,Jude Bellingham,England,1.210354,0.138843,0.051546,1.696631,0.6335,0.653061,0.923803
30,Anthony Gordon,England,0.923955,0.075188,0.051502,1.696631,0.6335,0.387755,0.792476
39,Declan Rice,England,0.889164,0.051282,0.027650,1.696631,0.6335,0.387755,0.809831
28,Bukayo Saka,England,0.824310,0.224719,0.136364,1.696631,0.6335,0.897959,0.521574
19,Alexis Mac Allister,Argentina,0.736784,0.181240,0.070000,1.218748,0.4373,0.306122,0.940338
61,Michael Olise,France,0.731314,0.107143,0.071429,1.235808,0.6335,0.897959,0.904956
79,Rodri,Spain,0.655102,0.220096,0.100000,1.037329,0.4373,0.489796,0.914046
55,Adrien Rabiot,France,0.619669,0.084848,0.066667,1.235808,0.6335,0.265306,0.813328
57,Manu Koné,France,0.592679,0.124352,0.043689,1.235808,0.6335,0.204082,0.717856


In [231]:
raw_cc = (
    0.25 * df["recent_cc_per90"]
  + 0.25 * df["stat_CC"]
  + 0.25 * df["match_over_25_prob"]
  + 0.15 * df["anytime_scorer_prob"]
  + 0.10 * df["price"]
)

df["lambda_cc"] = -np.log(
    (1 - raw_cc).clip(lower=0.001)
)

df["expected_chances_created"] = (
    df["lambda_cc"].clip(lower=0)
    * df["prob_plays_60min"] * 2
)

df.loc[df["position"] != "MID", "expected_chances_created"] = 0.0

In [232]:
cols = [
    "name",
    "team",
    "position",
    "expected_chances_created",
    "cc_per90",
    "recent_cc_per90",
    "stat_CC",
    "match_over_25_prob",
    "anytime_scorer_prob",
    "price",
    "prob_plays_60min",
]

df[cols].sort_values("expected_chances_created", ascending=False).head(30)

,name,team,position,expected_chances_created,cc_per90,recent_cc_per90,stat_CC,match_over_25_prob,anytime_scorer_prob,price,prob_plays_60min
49,Ousmane Dembélé,France,MID,1.138565,0.421196,0.257576,0.375,0.6335,0.3546,1.000000,0.897456
61,Michael Olise,France,MID,1.091306,0.691964,0.000000,0.625,0.6335,0.3226,0.897959,0.904956
40,Jude Bellingham,England,MID,0.803682,0.384298,0.000000,0.375,0.6335,0.2353,0.653061,0.923803
80,Dani Olmo,Spain,MID,0.718618,0.553571,0.273092,0.375,0.4373,0.1923,0.530612,0.824477
70,Lamine Yamal,Spain,MID,0.671337,0.156566,0.125926,0.125,0.4373,0.2703,1.000000,0.895495
30,Anthony Gordon,England,MID,0.642245,0.388471,0.145923,0.250,0.6335,0.2469,0.387755,0.792476
39,Declan Rice,England,MID,0.601304,0.496795,0.000000,0.375,0.6335,0.1282,0.387755,0.809831
14,Rodrigo De Paul,Argentina,MID,0.537000,0.364706,0.402367,0.250,0.4373,0.1143,0.163265,0.735370
28,Bukayo Saka,England,MID,0.536096,0.580524,0.257576,0.250,0.6335,0.1786,0.897959,0.521574
38,Elliot Anderson,England,MID,0.499692,0.124398,0.000000,0.125,0.6335,0.1250,0.285714,0.923895


In [233]:
raw_sot = (
    0.35 * df["sot_per90"]
  + 0.15 * df["stat_GS"]
  + 0.20 * df["match_over_25_prob"]
  + 0.20 * df["anytime_scorer_prob"]
  + 0.10 * df["price"]
)

df["lambda_sot"] = -np.log(
    (1 - raw_sot).clip(lower=0.001)
)

df["expected_shots_on_target"] = (
    df["lambda_sot"].clip(lower=0)
    * df["prob_plays_60min"] * 2
)

df.loc[df["position"] != "FWD", "expected_shots_on_target"] = 0.0

In [234]:
cols = [
    "name",
    "team",
    "position",
    "expected_shots_on_target",
    "sot_per90",
    "goal_rate",
    "stat_GS",
    "match_over_25_prob",
    "anytime_scorer_prob",
    "price",
    "prob_plays_60min",
]

df[cols].sort_values("expected_shots_on_target", ascending=False).head(30)

,name,team,position,expected_shots_on_target,sot_per90,goal_rate,stat_GS,match_over_25_prob,anytime_scorer_prob,price,prob_plays_60min
48,Kylian Mbappé,France,FWD,2.354337,0.656250,0.789474,1.000,0.6335,0.5814,1.000000,0.917843
5,Lionel Messi,Argentina,FWD,1.818101,0.609677,0.774194,1.000,0.4373,0.3817,0.923077,0.940797
27,Harry Kane,England,FWD,1.535962,0.385911,0.551302,0.750,0.6335,0.4132,1.000000,0.943489
72,Mikel Oyarzabal,Spain,FWD,1.125561,0.425414,0.552486,0.625,0.4373,0.3636,0.630769,0.897325
6,Julián Alvarez,Argentina,FWD,0.643142,0.210421,0.120240,0.125,0.4373,0.2632,0.707692,0.889910
91,Jean-Philippe Mateta,France,FWD,0.395375,1.000000,0.000000,0.000,0.6335,0.4082,0.384615,0.217639
86,Lautaro Martínez,Argentina,FWD,0.340729,0.270096,0.578778,0.375,0.4373,0.2941,0.738462,0.367576
71,Ferran Torres,Spain,FWD,0.282420,0.186667,0.000000,0.000,0.4373,0.3226,0.584615,0.437652
32,Noni Madueke,England,FWD,0.212827,0.072917,0.000000,0.000,0.6335,0.1887,0.323077,0.423327
95,Ollie Watkins,England,FWD,0.106424,0.000000,0.000000,0.000,0.6335,0.4405,0.600000,0.165610


In [235]:
df["expected_qualification_points"] = df["team_qualify_prob"] * 0

In [236]:
cols = [
    "name",
    "team",
    "position",
    "expected_qualification_points",
    "team_qualify_prob",
    "price",
    "prob_plays_60min",
]

df[cols].sort_values("team_qualify_prob", ascending=False).groupby("team").head(3).head(30)

,name,team,position,expected_qualification_points,team_qualify_prob,price,prob_plays_60min
0,Cristian Romero,Argentina,DEF,NaN,NaN,0.600000,0.904898
1,Marcos Senesi,Argentina,DEF,NaN,NaN,0.466667,0.101928
2,Nahuel Molina,Argentina,DEF,NaN,NaN,0.266667,0.806223
21,Nico O'Reilly,England,DEF,NaN,NaN,0.466667,0.746648
22,Marc Guéhi,England,DEF,NaN,NaN,0.733333,0.932549
23,Dan Burn,England,DEF,NaN,NaN,0.333333,0.140838
41,Theo Hernández,France,DEF,NaN,NaN,0.666667,0.261642
42,Maxence Lacroix,France,DEF,NaN,NaN,0.333333,0.253040
43,Dayot Upamecano,France,DEF,NaN,NaN,0.866667,0.923999
62,Yéremy Pino,Spain,MID,NaN,NaN,0.163265,0.118441


In [237]:
df.to_csv(OUT_CSV, index=False)

print(f"\nSaved {OUT_CSV} {df.shape[0]} rows x {df.shape[1]} cols")


Saved fantasy_enriched.csv 103 rows x 283 cols


In [238]:
BUDGET = 105.0
MAX_PER_COUNTRY =7
SQUAD_SIZE = 15
XI_SIZE = 11
BENCH_WEIGHT = 0.9

def compute_total_expected_points(df):
    df = df.copy()

    cs_value = {"GK": 5, "DEF": 5, "MID": 1, "FWD": 0}
    goal_value = {"GK": 9, "DEF": 7, "MID": 6, "FWD": 5}

    df["goal_pts_value"] = df["position"].map(goal_value)
    df["cs_pts_value"] = df["position"].map(cs_value)

    # appearance: 1pt for playing any, +1 if 60+
    e_appearance = df["prob_plays_any"] * 1 + df["prob_plays_60min"] * 1

    # goal
    e_goal = df["prob_scores"] * df["goal_pts_value"]

    # assist
    e_assist = df["expected_assists"] * 3

    # clean sheet: gated by 60min, value depends on position
    e_cs = df["prob_clean_sheet"] * df["cs_pts_value"]

    # goals conceded penalty: GK and DEF only
    e_gc = df["expected_gc_penalty"].copy()
    e_gc[~df["position"].isin(["GK", "DEF"])] = 0.0

    # saves bonus: GK only, every 3 saves = +1
    e_saves = (df["expected_saves"] / 3).copy()
    e_saves[df["position"] != "GK"] = 0.0

    # penalty save: GK only
    e_pen_save = (df["prob_penalty_save"] * 3).copy()
    e_pen_save[df["position"] != "GK"] = 0.0

    # tackles bonus: MID only, every 3 tackles = +1
    e_tackles = (df["expected_tackles"] / 3).copy()
    e_tackles[df["position"] != "MID"] = 0.0

    # big chances created: MID only, every 2 = +1
    e_cc = (df["expected_chances_created"] / 2).copy()
    e_cc[df["position"] != "MID"] = 0.0

    # shots on target: FWD only, every 2 = +1
    e_sot = (df["expected_shots_on_target"] / 2).copy()
    e_sot[df["position"] != "FWD"] = 0.0

    # yellow card
    e_yc = df["prob_yellow_card"] * -1

    # red card
    e_rc = df["prob_red_card"] * -2

    # own goal
    e_og = df["prob_own_goal"] * -2

    # penalty won
    e_pw = df["prob_pen_won"] * 2

    # penalty committed
    e_pc = df["prob_penalty_committed"] * -1

    # qualification bonus: +2 per player in XI who advances, requires playing
    # captain's quali bonus is NOT doubled per rules

    # scouting bonus: +2 if differential==1 AND player scores >4 base pts
    # model: use Poisson confidence interval on expected base points
    # base pts = everything except scouting and captain multiplier
    base_ep = (
        e_appearance + e_goal + e_assist + e_cs + e_gc +
        e_saves + e_pen_save + e_tackles + e_cc + e_sot +
        e_yc + e_rc + e_og + e_pw + e_pc  
    )

    # Poisson 90% CI: lower = ppf(0.05, mu), upper = ppf(0.95, mu)
    mu = base_ep.clip(lower=0.01)
    lower_ci = pd.Series(poisson.ppf(0.05, mu), index=df.index)
    upper_ci = pd.Series(poisson.ppf(0.95, mu), index=df.index)

    # if lower CI > 4: full +2 expected
    # if upper CI < 4: 0
    # if CI straddles 4: scale by P(X>4 | mu) * 2
    p_exceeds_4 = pd.Series(1 - poisson.cdf(4, mu), index=df.index)

    scouting_ep = pd.Series(0.0, index=df.index)
    eligible = df["differential"] == 1
    full_bonus = eligible & (lower_ci > 4)
    no_bonus = eligible & (upper_ci < 4)
    partial = eligible & ~full_bonus & ~no_bonus

    scouting_ep[full_bonus] = 2.0
    scouting_ep[no_bonus] = 0.0
    scouting_ep[partial] = p_exceeds_4[partial] * 2.0

    df["e_appearance"] = e_appearance
    df["e_goal"] = e_goal
    df["e_assist"] = e_assist
    df["e_cs"] = e_cs
    df["e_gc"] = e_gc
    df["e_saves"] = e_saves
    df["e_pen_save"] = e_pen_save
    df["e_tackles"] = e_tackles
    df["e_cc"] = e_cc
    df["e_sot"] = e_sot
    df["e_yc"] = e_yc
    df["e_rc"] = e_rc
    df["e_og"] = e_og
    df["e_pw"] = e_pw
    df["e_pc"] = e_pc
    df["e_scouting"] = scouting_ep

    df["expected_points"] = base_ep + scouting_ep

    return df

In [239]:

def optimize_squad(df, budget=BUDGET, max_per_country=MAX_PER_COUNTRY):
    idx = df.index.tolist()

    x = LpVariable.dicts("squad", idx, cat="Binary")
    s = LpVariable.dicts("xi", idx, cat="Binary")
    b = LpVariable.dicts("bench", idx, cat="Binary")    
    cap = LpVariable.dicts("cap", idx, cat="Binary")

    model = LpProblem("fantasy", LpMaximize)

    
    model += lpSum(
        df.loc[i, "expected_points"] * s[i]
        + df.loc[i, "expected_points"] * cap[i]
        + df.loc[i, "expected_points"] * BENCH_WEIGHT * b[i]
        for i in idx
    )

    # squad = 15, xi = 11, bench = 4
    model += lpSum(x[i] for i in idx) == 15
    model += lpSum(s[i] for i in idx) == 11
    model += lpSum(b[i] for i in idx) == 4

    for i in idx:
        model += s[i] + b[i] == x[i]

    # budget
    model += lpSum(df.loc[i, "price_raw"] * x[i] for i in idx) <= budget

    # squad composition: 2 GK, 5 DEF, 5 MID, 3 FWD
    for pos, count in [("GK", 2), ("DEF", 5), ("MID", 5), ("FWD", 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(x[i] for i in p_idx) == count

    # country limit
    for country in df["team"].unique():
        c_idx = df[df["team"] == country].index.tolist()
        model += lpSum(x[i] for i in c_idx) <= max_per_country


    # max 2 GK+DEF combined from the same country
    for country in df["team"].unique():
        gkdef_idx = df[(df["team"] == country) & (df["position"].isin(["GK", "DEF"]))].index.tolist()
        model += lpSum(x[i] for i in gkdef_idx) <= 2

    # XI formation: 1 GK starts, valid outfield formation
    gk_idx = df[df["position"] == "GK"].index.tolist()
    model += lpSum(s[i] for i in gk_idx) == 1
    model += lpSum(b[i] for i in gk_idx) == 1

    for pos, lo, hi in [("DEF", 3, 5), ("MID", 3, 5), ("FWD", 1, 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(s[i] for i in p_idx) >= lo
        model += lpSum(s[i] for i in p_idx) <= hi

    # captain: exactly 1, must be in XI
    model += lpSum(cap[i] for i in idx) == 1
    for i in idx:
        model += cap[i] <= s[i]

    model.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[model.status] != "Optimal":
        print(f"Solver status: {LpStatus[model.status]}")
        return None

    in_squad = [i for i in idx if value(x[i]) > 0.5]
    in_xi = [i for i in idx if value(s[i]) > 0.5]
    on_bench = [i for i in idx if value(b[i]) > 0.5]
    captain = next(i for i in idx if value(cap[i]) > 0.5)

    return {
        "squad": df.loc[in_squad].copy(),
        "xi": df.loc[in_xi].copy(),
        "bench": df.loc[on_bench].copy(),
        "captain": captain,
        "total_ep": value(model.objective),
        "cost": df.loc[in_squad, "price_raw"].sum(),
    }

In [240]:
for col in df.select_dtypes("number").columns:
    bad = df[col].isna().sum() + np.isinf(df[col]).sum()
    if bad > 0:
        print(f"{col}: {bad} NaN/inf")

next_fixture: 103 NaN/inf
betano_match_score: 13 NaN/inf
clubs_match_dist: 13 NaN/inf
team_qualify_prob: 103 NaN/inf
expected_qualification_points: 103 NaN/inf


In [241]:
def optimize_ideal_xi(df, max_per_country=MAX_PER_COUNTRY):
    # best possible 11 ignoring bench constraint, just 11 players
    idx = df.index.tolist()

    s = LpVariable.dicts("xi", idx, cat="Binary")
    cap = LpVariable.dicts("cap", idx, cat="Binary")

    model = LpProblem("ideal_xi", LpMaximize)

    # optimize_ideal_xi objective — NO bench term
    model += lpSum(
        df.loc[i, "expected_points"] * s[i]
        + df.loc[i, "expected_points"] * cap[i]
        for i in idx
    )

    model += lpSum(s[i] for i in idx) == 11

    # no budget constraint for ideal XI comparison
    # country limit still applies
    for country in df["team"].unique():
        c_idx = df[df["team"] == country].index.tolist()
        model += lpSum(s[i] for i in c_idx) <= max_per_country

    gk_idx = df[df["position"] == "GK"].index.tolist()
    model += lpSum(s[i] for i in gk_idx) == 1

    for pos, lo, hi in [("DEF", 3, 5), ("MID", 3, 5), ("FWD", 1, 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(s[i] for i in p_idx) >= lo
        model += lpSum(s[i] for i in p_idx) <= hi

    model += lpSum(cap[i] for i in idx) == 1
    for i in idx:
        model += cap[i] <= s[i]

    model.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[model.status] != "Optimal":
        return None

    in_xi = [i for i in idx if value(s[i]) > 0.5]
    captain = next(i for i in idx if value(cap[i]) > 0.5)

    return {
        "xi": df.loc[in_xi].copy(),
        "captain": captain,
        "total_ep": value(model.objective),
    }


def print_team(result, ideal=None):
    pos_order = {"GK": 0, "DEF": 1, "MID": 2, "FWD": 3}
    cap = result["captain"]

    xi = result["xi"].copy()
    xi["_ord"] = xi["position"].map(pos_order)
    xi = xi.sort_values(["_ord", "expected_points"], ascending=[True, False])

    bench = result["bench"].copy()
    bench["_ord"] = bench["position"].map(pos_order)
    bench = bench.sort_values(["_ord", "expected_points"], ascending=[True, False])

    print(f"\nExpected points: {result['total_ep']:.2f}   Cost: ${result['cost']:.1f}M\n")

    print("Starting XI")
    for i, row in xi.iterrows():
        tag = " [C]" if i == cap else ""
        scout = " [SCOUT]" if row.get("differential", 0) == 1 else ""
        print(f"  {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}{tag}{scout}")

    print("\nBench")
    for rank, (i, row) in enumerate(bench.iterrows(), 1):
        print(f"  [{rank}] {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}")

    print(f"\nCountry breakdown: {dict(result['squad']['team'].value_counts())}")

    if ideal:
        ideal_xi = ideal["xi"].copy()
        ideal_xi["_ord"] = ideal_xi["position"].map(pos_order)
        ideal_xi = ideal_xi.sort_values(["_ord", "expected_points"], ascending=[True, False])
        ideal_cap = ideal["captain"]

        print(f"\nIdeal XI (no budget constraint, 11 only, EP: {ideal['total_ep']:.2f})")
        for i, row in ideal_xi.iterrows():
            tag = " [C]" if i == ideal_cap else ""
            in_squad = i in result["squad"].index
            flag = "" if in_squad else " [NOT IN SQUAD]"
            print(f"  {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}{tag}{flag}")
            
if __name__ == "__main__":
    df = pd.read_csv("fantasy_enriched.csv")
    df = df[df["status"] == "playing"].copy()
    df = df.reset_index(drop=True)

    df = compute_total_expected_points(df)

    result = optimize_squad(df, budget=BUDGET, max_per_country=MAX_PER_COUNTRY)
    ideal = optimize_ideal_xi(df, max_per_country=MAX_PER_COUNTRY)

    print_team(result, ideal)


Expected points: 119.43   Cost: $104.9M

Starting XI
  GK   Unai Simón                   Spain                  $5.0  EP:3.80
  DEF  Marc Cucurella               Spain                  $5.1  EP:5.76
  DEF  Jules Koundé                 France                 $5.4  EP:4.43
  DEF  Dayot Upamecano              France                 $5.3  EP:4.21
  MID  Michael Olise                France                 $9.5  EP:10.92
  MID  Ousmane Dembélé              France                 $10.0  EP:10.22
  MID  Jude Bellingham              England                $8.3  EP:9.07
  MID  Anthony Gordon               England                $7.0  EP:6.55
  FWD  Lionel Messi                 Argentina              $10.0  EP:14.60 [C]
  FWD  Kylian Mbappé                France                 $10.5  EP:13.19
  FWD  Mikel Oyarzabal              Spain                  $8.1  EP:7.71

Bench
  [1] GK   Jordan Pickford              England                $4.8  EP:2.64
  [2] DEF  Cristian Romero              Argentin

In [242]:
def optimize_with_transfers(df, current_team_names, free_transfers=5, budget=BUDGET, max_per_country=MAX_PER_COUNTRY):

    current_idx = set()
    unmatched = []
    for name in current_team_names:
        match = df[df["name"].str.lower().str.strip() == name.lower().strip()]
        if len(match) == 0:
            match = df[df["name"].str.lower().str.contains(name.lower().strip(), na=False)]
        if len(match) > 0:
            current_idx.add(match.index[0])
        else:
            unmatched.append(name)

  
    if unmatched:
        print(f"Eliminated players: {unmatched}")

    forced_transfers = len(unmatched)

    idx = df.index.tolist()

    x         = LpVariable.dicts("squad", idx, cat="Binary")
    s         = LpVariable.dicts("xi",    idx, cat="Binary")
    b         = LpVariable.dicts("bench", idx, cat="Binary")
    cap       = LpVariable.dicts("cap",   idx, cat="Binary")
    transfer_out = LpVariable.dicts("out", idx, cat="Binary")

    model = LpProblem("fantasy_transfers", LpMaximize)

    transfers_made = lpSum(transfer_out[i] for i in idx)   # voluntary drops only

    n_extra = LpVariable("n_extra", lowBound=0)

    # CHANGE 2: total transfers = voluntary + forced. Penalty fires when
    # (voluntary + forced) > free_transfers, not just voluntary > free_transfers.
    model += n_extra >= transfers_made + forced_transfers - free_transfers

    model += (
        lpSum(
            df.loc[i, "expected_points"] * s[i]
            + df.loc[i, "expected_points"] * cap[i]
            + df.loc[i, "expected_points"] * BENCH_WEIGHT * b[i]
            for i in idx
        )
        - 3 * n_extra
    )

    for i in idx:
        if i in current_idx:
            model += transfer_out[i] == 1 - x[i]
        else:
            model += transfer_out[i] == 0

    model += lpSum(x[i] for i in idx) == 15
    model += lpSum(s[i] for i in idx) == 11
    model += lpSum(b[i] for i in idx) == 4

    for i in idx:
        model += s[i] + b[i] == x[i]

    model += lpSum(df.loc[i, "price_raw"] * x[i] for i in idx) <= budget

    for pos, count in [("GK", 2), ("DEF", 5), ("MID", 5), ("FWD", 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(x[i] for i in p_idx) == count

    for country in df["team"].unique():
        c_idx = df[df["team"] == country].index.tolist()
        model += lpSum(x[i] for i in c_idx) <= max_per_country

    gk_idx = df[df["position"] == "GK"].index.tolist()
    model += lpSum(s[i] for i in gk_idx) == 1
    model += lpSum(b[i] for i in gk_idx) == 1

    for pos, lo, hi in [("DEF", 3, 5), ("MID", 3, 5), ("FWD", 1, 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(s[i] for i in p_idx) >= lo
        model += lpSum(s[i] for i in p_idx) <= hi

    model += lpSum(cap[i] for i in idx) == 1
    for i in idx:
        model += cap[i] <= s[i]

    model.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[model.status] != "Optimal":
        print(f"Solver status: {LpStatus[model.status]}")
        return None

    in_squad  = [i for i in idx if value(x[i]) > 0.5]
    in_xi     = [i for i in idx if value(s[i]) > 0.5]
    on_bench  = [i for i in idx if value(b[i]) > 0.5]
    captain   = next(i for i in idx if value(cap[i]) > 0.5)

    new_squad_idx  = set(in_squad)
    voluntary_out  = [i for i in current_idx if i not in new_squad_idx]
    transfers_in   = [i for i in new_squad_idx if i not in current_idx]

    # CHANGE 3: total = voluntary drops of matched players + forced drops of eliminated.
    n_transfers = len(voluntary_out) + forced_transfers
    penalty     = max(0, n_transfers - free_transfers) * 3

    return {
        "squad":         df.loc[in_squad].copy(),
        "xi":            df.loc[in_xi].copy(),
        "bench":         df.loc[on_bench].copy(),
        "captain":       captain,
        "total_ep":      value(model.objective),
        "cost":          df.loc[in_squad, "price_raw"].sum(),
        "transfers_out": df.loc[voluntary_out, "name"].tolist() + unmatched,  # eliminated shown in OUT
        "transfers_in":  df.loc[transfers_in, "name"].tolist(),
        "n_transfers":   n_transfers,
        "penalty":       penalty,
        "forced_out":    unmatched,   # separate field for clarity
    }

def print_transfers(result):
    if result is None:
        print("No solution.")
        return

    print(f"\nTransfers made: {result['n_transfers']} ({result['n_transfers'] - 5} penalised at -3 each)" if result['n_transfers'] > 5 else f"\nTransfers made: {result['n_transfers']} (all free)")
    print(f"Point penalty: -{result['penalty']}")

    if result["transfers_out"]:
        print("\nOUT:")
        for name in result["transfers_out"]:
            print(f"  {name}")
        print("IN:")
        for name in result["transfers_in"]:
            print(f"  {name}")

    print_team(result)


if __name__ == "__main__":
    df = pd.read_csv("fantasy_enriched.csv")
    df = df[df["status"] == "playing"].copy()
    df = df.reset_index(drop=True)
    df = compute_total_expected_points(df)

    current_team = [
        "Emiliano Martínez", "Lisandro Martínez", "Achraf Hakimi", "Nico O'Reilly", "Marc Cucurella",
        "Ousmane Dembélé", "Michael Olise", "Unai Simón", "Jude Bellingham", "Kylian Mbappé", "Mikel Oyarzabal",
        "Dani Olmo", "Facundo Medina", "Lionel Messi", "Brahim Díaz"
    ]

    result = optimize_with_transfers(df, current_team, free_transfers=5, budget=BUDGET, max_per_country=MAX_PER_COUNTRY)
    ideal = optimize_ideal_xi(df, max_per_country=MAX_PER_COUNTRY)
    print_transfers(result)

    print("\nIdeal XI for comparison:")
    if ideal:
        pos_order = {"GK": 0, "DEF": 1, "MID": 2, "FWD": 3}
        xi = ideal["xi"].copy()
        xi["_ord"] = xi["position"].map(pos_order)
        xi = xi.sort_values(["_ord", "expected_points"], ascending=[True, False])
        cap = ideal["captain"]
        for i, row in xi.iterrows():
            tag = " [C]" if i == cap else ""
            in_squad = i in result["squad"].index if result else False
            flag = "" if in_squad else " [NOT IN SQUAD]"
            print(f"  {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}{tag}{flag}")

Eliminated players: ['Achraf Hakimi', 'Brahim Díaz']

Transfers made: 5 (all free)
Point penalty: -0

OUT:
  Dani Olmo
  Facundo Medina
  Nico O'Reilly
  Achraf Hakimi
  Brahim Díaz
IN:
  Cristian Romero
  Aymeric Laporte
  Pau Cubarsí
  Adrien Rabiot
  Anthony Gordon

Expected points: 120.08   Cost: $104.9M

Starting XI
  GK   Unai Simón                   Spain                  $5.0  EP:3.80
  DEF  Marc Cucurella               Spain                  $5.1  EP:5.76
  DEF  Aymeric Laporte              Spain                  $5.5  EP:4.60
  DEF  Pau Cubarsí                  Spain                  $5.0  EP:4.18
  MID  Michael Olise                France                 $9.5  EP:10.92
  MID  Ousmane Dembélé              France                 $10.0  EP:10.22
  MID  Jude Bellingham              England                $8.3  EP:9.07
  MID  Anthony Gordon               England                $7.0  EP:6.55
  FWD  Lionel Messi                 Argentina              $10.0  EP:14.60 [C]
  FWD  Kyli

In [243]:
def optimize_with_transfers(df, current_team_names, free_transfers=5, budget=BUDGET, max_per_country=MAX_PER_COUNTRY):

    current_idx = set()
    unmatched = []
    for name in current_team_names:
        match = df[df["name"].str.lower().str.strip() == name.lower().strip()]
        if len(match) == 0:
            match = df[df["name"].str.lower().str.contains(name.lower().strip(), na=False)]
        if len(match) > 0:
            current_idx.add(match.index[0])
        else:
            unmatched.append(name)

  
    if unmatched:
        print(f"Eliminated players: {unmatched}")

    forced_transfers = len(unmatched)

    idx = df.index.tolist()

    x         = LpVariable.dicts("squad", idx, cat="Binary")
    s         = LpVariable.dicts("xi",    idx, cat="Binary")
    b         = LpVariable.dicts("bench", idx, cat="Binary")
    cap       = LpVariable.dicts("cap",   idx, cat="Binary")
    transfer_out = LpVariable.dicts("out", idx, cat="Binary")

    model = LpProblem("fantasy_transfers", LpMaximize)

    transfers_made = lpSum(transfer_out[i] for i in idx)   # voluntary drops only

    n_extra = LpVariable("n_extra", lowBound=0)

    # CHANGE 2: total transfers = voluntary + forced. Penalty fires when
    # (voluntary + forced) > free_transfers, not just voluntary > free_transfers.
    model += n_extra >= transfers_made + forced_transfers - free_transfers

    model += (
        lpSum(
            df.loc[i, "expected_points"] * s[i]
            + df.loc[i, "expected_points"] * cap[i]
            + df.loc[i, "expected_points"] * BENCH_WEIGHT * b[i]
            for i in idx
        )
        - 3 * n_extra
    )

    for i in idx:
        if i in current_idx:
            model += transfer_out[i] == 1 - x[i]
        else:
            model += transfer_out[i] == 0

    model += lpSum(x[i] for i in idx) == 15
    model += lpSum(s[i] for i in idx) == 11
    model += lpSum(b[i] for i in idx) == 4

    for i in idx:
        model += s[i] + b[i] == x[i]

    model += lpSum(df.loc[i, "price_raw"] * x[i] for i in idx) <= budget

    for pos, count in [("GK", 2), ("DEF", 5), ("MID", 5), ("FWD", 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(x[i] for i in p_idx) == count

    for country in df["team"].unique():
        c_idx = df[df["team"] == country].index.tolist()
        model += lpSum(x[i] for i in c_idx) <= max_per_country

    gk_idx = df[df["position"] == "GK"].index.tolist()
    model += lpSum(s[i] for i in gk_idx) == 1
    model += lpSum(b[i] for i in gk_idx) == 1

    for pos, lo, hi in [("DEF", 3, 5), ("MID", 3, 5), ("FWD", 1, 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(s[i] for i in p_idx) >= lo
        model += lpSum(s[i] for i in p_idx) <= hi

    model += lpSum(cap[i] for i in idx) == 1
    for i in idx:
        model += cap[i] <= s[i]

    model.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[model.status] != "Optimal":
        print(f"Solver status: {LpStatus[model.status]}")
        return None

    in_squad  = [i for i in idx if value(x[i]) > 0.5]
    in_xi     = [i for i in idx if value(s[i]) > 0.5]
    on_bench  = [i for i in idx if value(b[i]) > 0.5]
    captain   = next(i for i in idx if value(cap[i]) > 0.5)

    new_squad_idx  = set(in_squad)
    voluntary_out  = [i for i in current_idx if i not in new_squad_idx]
    transfers_in   = [i for i in new_squad_idx if i not in current_idx]

    # CHANGE 3: total = voluntary drops of matched players + forced drops of eliminated.
    n_transfers = len(voluntary_out) + forced_transfers
    penalty     = max(0, n_transfers - free_transfers) * 3

    return {
        "squad":         df.loc[in_squad].copy(),
        "xi":            df.loc[in_xi].copy(),
        "bench":         df.loc[on_bench].copy(),
        "captain":       captain,
        "total_ep":      value(model.objective),
        "cost":          df.loc[in_squad, "price_raw"].sum(),
        "transfers_out": df.loc[voluntary_out, "name"].tolist() + unmatched,  # eliminated shown in OUT
        "transfers_in":  df.loc[transfers_in, "name"].tolist(),
        "n_transfers":   n_transfers,
        "penalty":       penalty,
        "forced_out":    unmatched,   # separate field for clarity
    }

def print_transfers(result):
    if result is None:
        print("No solution.")
        return

    print(f"\nTransfers made: {result['n_transfers']} ({result['n_transfers'] - 5} penalised at -3 each)" if result['n_transfers'] > 5 else f"\nTransfers made: {result['n_transfers']} (all free)")
    print(f"Point penalty: -{result['penalty']}")

    if result["transfers_out"]:
        print("\nOUT:")
        for name in result["transfers_out"]:
            print(f"  {name}")
        print("IN:")
        for name in result["transfers_in"]:
            print(f"  {name}")

    print_team(result)


if __name__ == "__main__":
    df = pd.read_csv("fantasy_enriched.csv")
    df = df[df["status"] == "playing"].copy()
    df = df.reset_index(drop=True)
    df = compute_total_expected_points(df)

    current_team = [
        "Mike Maignan", "Lisandro Martínez", "Dayot Upamecano", "Nico O'Reilly", "Marc Cucurella",
        "Ousmane Dembélé", "Michael Olise", "Jude Bellingham", "Harry Kane", "Kylian Mbappé", "Álex Baena",
        "Unai Simón", "Cristian Romero", "Lionel Messi", "Patrick Berg"
    ]

    result = optimize_with_transfers(df, current_team, free_transfers=5, budget=BUDGET, max_per_country=MAX_PER_COUNTRY)
    ideal = optimize_ideal_xi(df, max_per_country=MAX_PER_COUNTRY)
    print_transfers(result)

Eliminated players: ['Patrick Berg']

Transfers made: 5 (all free)
Point penalty: -0

OUT:
  Dayot Upamecano
  Nico O'Reilly
  Harry Kane
  Álex Baena
  Patrick Berg
IN:
  Aymeric Laporte
  Pau Cubarsí
  Mikel Oyarzabal
  Adrien Rabiot
  Anthony Gordon

Expected points: 120.09   Cost: $104.9M

Starting XI
  GK   Unai Simón                   Spain                  $5.0  EP:3.80
  DEF  Marc Cucurella               Spain                  $5.1  EP:5.76
  DEF  Aymeric Laporte              Spain                  $5.5  EP:4.60
  DEF  Pau Cubarsí                  Spain                  $5.0  EP:4.18
  MID  Michael Olise                France                 $9.5  EP:10.92
  MID  Ousmane Dembélé              France                 $10.0  EP:10.22
  MID  Jude Bellingham              England                $8.3  EP:9.07
  MID  Anthony Gordon               England                $7.0  EP:6.55
  FWD  Lionel Messi                 Argentina              $10.0  EP:14.60 [C]
  FWD  Kylian Mbappé       

In [244]:
for pos in ["GK", "DEF", "MID", "FWD"]:
    print(f"\n-{pos}")
    print(df[df["position"] == pos].nlargest(10, "expected_points")[["name", "team", "price_raw", "expected_points"]].to_string(index=False))


-GK
             name      team  price_raw  expected_points
       Unai Simón     Spain        5.0         3.800963
     Mike Maignan    France        5.0         3.222669
Emiliano Martínez Argentina        5.0         3.209109
  Jordan Pickford   England        4.8         2.640925
       David Raya     Spain        5.0         0.621179
   Gerónimo Rulli Argentina        4.5         0.496508
      Joan García     Spain        4.0         0.478058
     Robin Risser    France        3.5         0.470305
      Brice Samba    France        4.5         0.465744
       Juan Musso Argentina        4.3         0.456443

-DEF
             name      team  price_raw  expected_points
   Marc Cucurella     Spain        5.1         5.761274
  Aymeric Laporte     Spain        5.5         4.600552
     Jules Koundé    France        5.4         4.427621
      Pedro Porro     Spain        5.5         4.343051
  Dayot Upamecano    France        5.3         4.210752
      Pau Cubarsí     Spain        5.

In [245]:
player1 = "Harry Kane"
player2 = "Mikel Oyarzabal"

cols = [
    "name", "team", "position", "price_raw", "expected_points",
    "e_appearance", "e_goal", "e_assist", "e_cs", "e_gc",
    "e_saves", "e_pen_save", "e_tackles", "e_cc", "e_sot",
    "e_yc", "e_rc", "e_og", "e_pw", "e_pc", "e_scouting"
]

comparison = df.loc[
    df["name"].isin([player1, player2]),
    cols
].set_index("name").T

print(comparison)

name            Harry Kane Mikel Oyarzabal
team               England           Spain
position               FWD             FWD
price_raw             10.5             8.1
expected_points   8.861113        7.705808
e_appearance      1.875963        1.801515
e_goal            5.130871         4.13245
e_assist          1.295018        1.180006
e_cs                   0.0             0.0
e_gc                   0.0             0.0
e_saves                0.0             0.0
e_pen_save             0.0             0.0
e_tackles              0.0             0.0
e_cc                   0.0             0.0
e_sot             0.767981         0.56278
e_yc             -0.228222         -0.0438
e_rc             -0.054386       -0.015429
e_og             -0.028871       -0.006292
e_pw              0.138669        0.106064
e_pc             -0.035911       -0.011487
e_scouting             0.0             0.0
